# Transcript Appendix: EDA and Model Evidence

This appendix expands the transcript portion of the final prototype evidence notebook. The master reference remains `notebooks/00_final_prototype_evidence_notebook.ipynb`.

Purpose: document transcript dataset readiness, 80/20 text training logic, saved transcript model metrics, optional transformer evidence, and dashboard traceability without rerunning training.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def project_path(relative: str) -> Path:
    return ROOT / relative

def load_json(relative: str) -> dict:
    path = project_path(relative)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def status(path: Path) -> str:
    return "PASS" if path.exists() else "MISSING"

ROOT

## Dashboard Input, Visible Output, and Report Handoff

This appendix starts from the transcript/audio experience a user can see. The important handoff is: user evidence becomes transcript text, model/rule outputs become visible risk evidence, and selected results become report rows.

In [ ]:
dashboard_io_rows = [
    {
        "Dashboard step": "Provide transcript evidence",
        "User-visible input": "Paste transcript text, upload a transcript file, or use audio that can be transcribed.",
        "User-visible output": "Transcript preview and editable text area used for analysis.",
        "Report/history fields": "preview, raw_input, source_name where available",
        "Documentation note": "This preserves the exact text the text model inspected.",
    },
    {
        "Dashboard step": "Choose analysis settings",
        "User-visible input": "Select transcript scam models and, for audio, choose Whisper settings.",
        "User-visible output": "Selected model labels, recommended model, and model agreement summary.",
        "Report/history fields": "model/model_name, prediction, confidence",
        "Documentation note": "Whisper is transcription support; the scam classifier evaluates the resulting text.",
    },
    {
        "Dashboard step": "Review transcript result",
        "User-visible input": "No extra input after analysis.",
        "User-visible output": "Final verdict, suspicious-risk score, confidence chart, model agreement table, rule indicators, highlighted terms, and explanation.",
        "Report/history fields": "prediction, confidence, flags, explanation, preview, raw_input",
        "Documentation note": "Transcript history stores more explanation detail than the email core record.",
    },
    {
        "Dashboard step": "Convert to report evidence",
        "User-visible input": "Open AI Report Generator and select saved Transcript evidence rows.",
        "User-visible output": "Report preview and TXT/PDF/DOCX download.",
        "Report/history fields": "scan_type, prediction, confidence, model_name, preview, flags, explanation, raw_input",
        "Documentation note": "This shows how spoken or written call evidence becomes a readable report item.",
    },
]

pd.DataFrame(dashboard_io_rows)

## Dataset Readiness

Transcript models focus on conversation text rather than email style. Scam-only transcript sources can support demonstrations, but binary model training needs both scam and non-scam examples.

In [ ]:
transcript_path = project_path("data/processed/transcript/transcript_dataset.csv")
transcript_df = pd.read_csv(transcript_path) if transcript_path.exists() else pd.DataFrame()

pd.DataFrame([
    {
        "Evidence item": "Processed transcript dataset",
        "Path": str(transcript_path.relative_to(ROOT)),
        "Status": status(transcript_path),
        "Rows": len(transcript_df),
        "Columns": len(transcript_df.columns),
        "Label column present": "label" in transcript_df.columns,
    }
])

In [ ]:
if not transcript_df.empty and "label" in transcript_df.columns:
    counts = transcript_df["label"].value_counts().sort_index()
    ax = counts.plot(kind="bar", figsize=(6, 4), color=["#16A34A", "#DC2626"])
    ax.set_title("Transcript Label Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
    plt.tight_layout()
else:
    print("Processed transcript dataset or label column is unavailable.")

## Model Selection Evidence

Transcript text uses a stratified 80/20 split in the training workflow. TF-IDF models such as Naive Bayes and SVM can be compared with optional transformer models such as DistilBERT when the artifact is available.

In [ ]:
transcript_metrics = load_json("reports/metrics/transcript_model_metrics.json")
selected_name = transcript_metrics.get("recommended_model") or transcript_metrics.get("top_validation_model")
rows = []
for model_name, metrics in transcript_metrics.get("models", {}).items():
    rows.append({
        "Model": model_name,
        "Recommended": "Yes" if model_name == selected_name else "No",
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
        "ROC-AUC": metrics.get("roc_auc"),
    })

metric_df = pd.DataFrame(rows)
metric_df.sort_values("F1", ascending=False) if not metric_df.empty else pd.DataFrame([{"Status": "No saved transcript metric rows found"}])

In [ ]:
if not metric_df.empty and "F1" in metric_df.columns:
    plot_df = metric_df.sort_values("F1", ascending=True)
    colors = ["#2563EB" if value == "Yes" else "#94A3B8" for value in plot_df["Recommended"]]
    ax = plot_df.plot.barh(x="Model", y="F1", figsize=(7, 4), color=colors, legend=False)
    ax.set_title("Transcript Candidate Model F1 Scores")
    ax.set_xlabel("F1 score")
    plt.tight_layout()
else:
    print("No transcript model metrics available for plotting.")

## Runtime Artifact And Source Traceability

Whisper or local transcription support should be explained as speech-to-text support. The scam classifier is the saved transcript model that evaluates the resulting text.

In [ ]:
artifact_paths = [
    "models/transcript_vectorizer.pkl",
    "models/transcript_nb.pkl",
    "models/transcript_svm.pkl",
    "models/transcript_distilbert",
    "reports/metrics/transcript_model_metrics.json",
    "app/transcript_tab.py",
    "src/training/transcript_trainer.py",
    "scripts/05_train_transcript_model.py",
]

pd.DataFrame([
    {"Path": relative, "Status": status(project_path(relative))}
    for relative in artifact_paths
])

## Reviewer Note

Transcript results can be sensitive to short or ambiguous wording, transcription quality, and class coverage in the dataset. Use the model output as evidence for review, not as the final judgement.